In [1]:
# Import necessary libraries for BERT
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import Trainer, TrainingArguments, BertTokenizer, BertForSequenceClassification
from torch.utils.data import Dataset
import torch
from transformers import AdamW


In [2]:
# Step 1: Load and preprocess the dataset
# Load datasets
fake_df = pd.read_csv('Fake.csv')
true_df = pd.read_csv('True.csv')

# Add labels
fake_df['label'] = 0
true_df['label'] = 1

# Combine and shuffle
combined_df = pd.concat([fake_df, true_df], axis=0).sample(frac=1, random_state=42).reset_index(drop=True)

# Create a content column
combined_df['content'] = combined_df['title'] + " " + combined_df['text']

# Drop missing values
combined_df = combined_df.dropna()

# Split data
X = combined_df['content']
y = combined_df['label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [3]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

In [4]:
# Step 2: Custom Dataset class
class NewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=512):
        self.encodings = tokenizer(texts, truncation=True, padding=True, max_length=max_length)
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels.iloc[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [5]:
train_dataset = NewsDataset(X_train.tolist(), y_train, tokenizer)
test_dataset = NewsDataset(X_test.tolist(), y_test, tokenizer)

from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False, num_workers=4)


In [6]:
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [7]:
# Training arguments for the BERT model
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
    eval_strategy="epoch",
    fp16=True,  # Mixed precision
    gradient_accumulation_steps=4,  # Accumulate gradients over 4 steps
    dataloader_num_workers=4  # Efficient data loading
)


In [8]:
# Trainer initialization
from transformers import DataCollatorWithPadding
data_collator = DataCollatorWithPadding(tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=data_collator  # Enable dynamic padding
)



In [9]:
import torch
print(torch.cuda.is_available())  # Should return True
print(torch.cuda.device_count())  # Check if multiple GPUs are available


True
1


In [ ]:
try:
    trainer.train()
except Exception as e:
    print(f"Training failed: {e}")


In [ ]:
# Evaluate the model
results = trainer.evaluate()

results

In [ ]:
# Step 9: Save the fine-tuned model
model.save_pretrained('./fine_tuned_bert')
tokenizer.save_pretrained('./fine_tuned_bert')




In [ ]:
# Step 10: Load the fine-tuned model for inference
fine_tuned_model = BertForSequenceClassification.from_pretrained('./fine_tuned_bert')
fine_tuned_tokenizer = BertTokenizer.from_pretrained('./fine_tuned_bert')



In [ ]:
# Example inference
inputs = fine_tuned_tokenizer("This is a test sentence.", return_tensors="pt")
outputs = fine_tuned_model(**inputs)
predictions = torch.argmax(outputs.logits, dim=-1)
print("Predicted class:", predictions.item())